# Diagnostics — wire contract, HEF audit, device probes

The companion to [walkthrough.ipynb](walkthrough.ipynb): everything needed
to look INSIDE the compiled result and debug the runtime layer by layer.
Three independent sections:

| Section | Needs a Hailo device? | Content |
|---|---|---|
| **A — Wire inputs** | no | rebuild by hand exactly what the genai runtime constructs host-side: attention-mask layout, RoPE tables, uint16/uint8 encodings |
| **B — Static HEF audit** | no | decode `model.hef` offline: network groups, embedded resources, hailo-config |
| **C — Device probes** | **yes** | drive real silicon through the low-level `InferModel` API |

Evidence backing every convention reproduced here:
[docs/findings/](../docs/findings/index.md).

## 0. Common setup (run once)

In [ ]:
import os

# Same constraint as the pipeline: legacy NumPy promotion before any numpy
# import (the DFC toolchain relies on it).
os.environ.setdefault("NPY_PROMOTION_STATE", "legacy")

import sys

import numpy as np

# Probe matplotlib against this environment's NumPy before relying on it:
# old matplotlib releases (< 3.9) crash on NumPy 2 ("np.Inf was removed").
# The notebooks still work without plots (textual renders are provided).
try:
    import matplotlib.pyplot as plt

    _probe_fig, _probe_ax = plt.subplots()
    _probe_ax.imshow(np.zeros((2, 2)))
    plt.tight_layout()
    plt.close(_probe_fig)
    HAVE_PLOT = True
except Exception as _exc:
    HAVE_PLOT = False
    print("matplotlib unusable here -- plot cells will degrade to text.")
    print("reason:", _exc)

# --- Geometry of the reference model (keep in sync with walkthrough step 0)
HIDDEN = 256
NHEAD = 16
NKVHEAD = 8
HD = HIDDEN // NHEAD          # 16
SEQ = 24                      # total KV-cache size / base scope length
PREFILL_SIZE = 16             # __prefill scope length
ROPE_THETA = 10000.0
VOCAB = 32000
NET_SCOPE = "ts25mpipe"
ALLOWED = 0.0
BLOCKED = -100.0

# Quantization params of the embedding input layer (a16_w16) -- read these
# from your own HAR/HEF when porting another model.
EMB_SCALE = 1.440075175196398e-05
EMB_ZP = 15923.0

# --- Artifacts produced by the walkthrough notebook ------------------------
REPO = os.path.dirname(os.getcwd()) if os.path.basename(os.getcwd()) == "notebooks" else os.getcwd()
WORKDIR = os.environ.get("DFC_WORKDIR", os.path.join(REPO, "workdir"))
HEF = os.path.join(WORKDIR, "model.hef")
WTE = os.path.join(WORKDIR, "wte.npy")
HF_REFS = os.path.join(WORKDIR, "hf_reference.npz")
DIAG_DIR = os.path.join(REPO, "runtime", "diagnostics")

print("repo    :", REPO)
print("workdir :", WORKDIR)
print("hef     :", HEF, "(exists)" if os.path.exists(HEF) else "(missing -- run walkthrough.ipynb first)")

## A · Attention mask — what the host really sends

Additive float mask (allowed = 0, blocked = −100), quantized on the wire to
uint8 (**allowed → 255, blocked → 0**). Head-tiling (`×NHEAD`) is done
host-side, exactly like `genai::prepare_attention_mask_input`.

Structure over the `CACHE_SIZE` columns — two blocks of rows:

- **block 1** (`layer_input_tokens_size − mask_cache_usage` rows):
  positions already fully in cache → everything allowed;
- **block 2** (`mask_cache_usage` rows):
  `[unused columns | freshly written cache columns | causal self-block]`.

Prefill (`__prefill`, everything fresh): cols `0..7` unused, cols `8..23`
causal lower-triangular. TBT step *k* (`__tbt`): `[unused 8−k][fresh k][self]`.

In [ ]:
def build_mask(layer_input_tokens_size, mask_cache_usage, cache_size, n_heads):
    """Replicate the runtime's mask structure for one network group.

    Returns shape [1, rows, n_heads * cache_size] (head-tiled).
    """
    block1_rows = max(layer_input_tokens_size - mask_cache_usage, 0)
    block2_rows = min(mask_cache_usage, layer_input_tokens_size)
    rows = block1_rows + block2_rows

    m = np.full((rows, cache_size), BLOCKED, dtype=np.float32)
    if block1_rows > 0:
        m[:block1_rows, :] = ALLOWED
    if block2_rows > 0:
        r0 = block1_rows
        unused_cols = cache_size - mask_cache_usage
        m[r0:, :unused_cols] = BLOCKED
        mid_cols = mask_cache_usage - block2_rows
        if mid_cols > 0:
            m[r0:, unused_cols:unused_cols + mid_cols] = ALLOWED
        right_col = unused_cols + mid_cols
        self_block = np.full((block2_rows, block2_rows), BLOCKED, dtype=np.float32)
        self_block[np.tril_indices(block2_rows, k=0)] = ALLOWED
        m[r0:, right_col:right_col + block2_rows] = self_block

    return np.tile(m[np.newaxis, :, :], (1, 1, n_heads)).astype(np.float32)


def render(head0):
    """Text view of one head tile: '#' = allowed, '.' = blocked."""
    return "\n".join("".join("#" if v == ALLOWED else "." for v in row) for row in head0)


# Prefill group: all PREFILL_SIZE rows are "fresh" (cache usage == prefill).
mask_prefill = build_mask(PREFILL_SIZE, PREFILL_SIZE, SEQ, NHEAD)

# TBT group at generation step k=3: 16 prefill columns + 3 fresh ones.
K = 3
mask_tbt_k = build_mask(SEQ, PREFILL_SIZE + K, SEQ, NHEAD)

print(f"prefill mask {mask_prefill[0].shape} (head 0 tile):")
print(render(mask_prefill[0]))
print()
print(f"tbt mask at step k={K} {mask_tbt_k[0].shape} (head 0 tile):")
print(render(mask_tbt_k[0]))

In [ ]:
# Same two masks as images. White = allowed, black = blocked.
if not HAVE_PLOT:
    print("(plot skipped -- the textual render above shows the same layout)")
else:
    fig, axes = plt.subplots(1, 2, figsize=(11, 4))
    axes[0].imshow(mask_prefill[0], cmap="gray_r", aspect="auto")
    axes[0].set_title(f"__prefill mask {mask_prefill[0].shape}")
    axes[1].imshow(mask_tbt_k[0], cmap="gray_r", aspect="auto")
    axes[1].set_title(f"__tbt mask at step k={K} {mask_tbt_k[0].shape}")
    for ax in axes:
        ax.set_xlabel("cache position (head 0 tile)")
        ax.set_ylabel("query row")
    plt.tight_layout()
    plt.show()

## A · RoPE tables + wire encodings

**RoPE** — the host computes cos/sin from integer positions against the
theta table `1/(θ^(arange(0,HD,2)/HD))` concatenated twice, tiled per head
group. Fix #2 asymmetry applies: K inputs are `θ_size × NKVHEAD` wide,
Q inputs `θ_size × NHEAD` wide.

**Embeddings** — fp32 rows quantized to uint16 codes:
`round(row / scale + zero_point)`, clipped to `[0, 65535]`.

**Mask** — additive float → raw uint8: allowed → 255, blocked → 0.

In [ ]:
theta = np.concatenate([
    1.0 / (ROPE_THETA ** (np.arange(0, HD, 2, dtype=np.float64) / HD)),
] * 2).astype(np.float32)


def build_rope(positions, groups, theta):
    """Cos/sin tables for `positions`, tiled across `groups` heads.

    theta is the doubled table (cos-half + sin-half): tiling along columns
    already yields the final wire width -- K = 16 x 8 = 128, Q = 16 x 16 = 256.
    Returns (cos, sin), each [1, len(positions), groups * theta.shape[0]]."""
    positions = np.asarray(list(positions))
    angles = np.outer(positions.astype(np.float64), theta.astype(np.float64))
    cos = np.tile(np.cos(angles), (1, groups)).reshape(1, len(positions), groups * theta.shape[0])
    sin = np.tile(np.sin(angles), (1, groups)).reshape(1, len(positions), groups * theta.shape[0])
    return cos.astype(np.float32), sin.astype(np.float32)


positions = np.arange(SEQ)
cos_k, sin_k = build_rope(positions, groups=NKVHEAD, theta=theta)   # input_layer3 / input_layer5
cos_q, sin_q = build_rope(positions, groups=NHEAD, theta=theta)     # input_layer4 / input_layer6
print("K cos/sin:", cos_k.shape, "   Q cos:", cos_q.shape, " <- fix #2 asymmetry")

if not HAVE_PLOT:
    print("(plot skipped)")
else:
    fig, axes = plt.subplots(1, 2, figsize=(11, 3))
    axes[0].imshow(cos_k[0], aspect="auto")
    axes[0].set_title("rope cos(position × θ), KV width (128)")
    axes[1].imshow(sin_k[0], aspect="auto")
    axes[1].set_title("rope sin(position × θ), KV width (128)")
    for ax in axes:
        ax.set_xlabel("tiled head-group width")
        ax.set_ylabel("position")
    plt.tight_layout()
    plt.show()

In [ ]:
def encode_embeddings_uint16(float_rows, scale, zp):
    codes = np.round(float_rows.astype(np.float64) / scale + zp)
    return np.clip(codes, 0, 65535).astype(np.uint16)


def encode_mask_uint8(float_mask, allowed=ALLOWED):
    return np.where(float_mask == allowed, 255, 0).astype(np.uint8)


def cosine(a, b):
    a = a.flatten().astype(np.float64)
    b = b.flatten().astype(np.float64)
    return float(np.dot(a, b) / (np.linalg.norm(a) * np.linalg.norm(b) + 1e-12))


# Mask -> wire values
wire_mask = encode_mask_uint8(mask_tbt_k)
print("mask wire values:", np.unique(wire_mask), "(255 = allowed, 0 = blocked)")

# Embedding row -> uint16 codes -> back to float: quantization is lossless
# to within the input-layer resolution (cosine ~ 1.0).
if os.path.exists(WTE):
    row = np.load(WTE)[7]                    # a real embedding row
    source = f"{WTE} (row 7)"
else:
    row = np.random.default_rng(0).normal(size=HIDDEN) * 0.02
    source = "random row (workdir/wte.npy missing)"
codes = encode_embeddings_uint16(row[np.newaxis, :], EMB_SCALE, EMB_ZP)
back = (codes.astype(np.float64)[0] - EMB_ZP) * EMB_SCALE
print(f"embedding row from {source}")
print("uint16 code range:", codes.min(), "..", codes.max())
print(f"encode->decode cosine vs fp32 row: {cosine(row, back):.6f}")

## B — Static HEF audit (offline)

Decodes `model.hef` without any device: network-group names/shapes/formats,
the four embedded external resources (embeddings.bin, tokenizer.json,
rope_theta_data.bin, hailo-config.json — located through their
zero-point–corrected offsets), and `hailortcli parse-hef` metadata.

Uses the repo's `runtime/diagnostics/hef_audit.py`, which parses the HEF
header + protobuf metadata directly (format from the public MIT-licensed
HailoRT source) and does bounded seek+reads only — never loads a multi-GB
HEF into RAM.

In [ ]:
try:
    import hailo_platform  # noqa: F401
    HAVE_HAILORT = True
except Exception as exc:
    HAVE_HAILORT = False
    print("hailo_platform not importable here -- skipping section B:", exc)

sys.path.insert(0, DIAG_DIR)

if HAVE_HAILORT and not os.path.exists(HEF):
    print("no HEF yet -- run walkthrough.ipynb first")

if HAVE_HAILORT and os.path.exists(HEF):
    from pprint import pprint

    from hef_audit import audit_hef

    report = audit_hef(HEF)
    for key, value in report.items():
        print(f"===== {key} =====")
        pprint(value, depth=2, width=110)

## C — Device probes (device host only)

Two probes against real silicon, driven through the low-level `InferModel`
API with hand-built inputs from section A:

1. **`generate_base_scope.py`** — greedy generation entirely inside the base
   scope (right-aligned prompt, no KV-cache duplication). Validated
   coherent on hardware; proves weights/RoPE/GQA/embeddings/lm_head/INT4 are
   all sound.
2. **`manual_prefill_tbt_test.py`** — prefill once, then step `__tbt` token
   by token, comparing hidden states and logits against captured reference
   tensors. This is the instrument that localized the open issue: `__prefill`
   never reads the cache and is quasi-exact, while `__tbt` cache reads come
   back with structurally truncated columns (~30% of the cached tensor
   returns zeroed regardless of what was written).

> ⚠️ The full `manual_prefill_tbt_test` needs a reference `.npz` built by its
> tap-capture harness (hidden-state clones inside the graph) — see
> `runtime/diagnostics/README.md`. The walkthrough's `hf_reference.npz`
> alone is not sufficient.

In [ ]:
dev_nodes = sorted([f"/dev/{p}" for p in os.listdir("/dev") if p.startswith("hailo")])
HAVE_DEVICE = HAVE_HAILORT and bool(dev_nodes) and os.path.exists(HEF)
if HAVE_DEVICE:
    print("Hailo device nodes:", dev_nodes)
elif not dev_nodes:
    print("No Hailo device node under /dev -- section C cells will skip.")
    print("(Run them on the device host; see docs/device-setup.md)")

In [ ]:
if not HAVE_DEVICE:
    print("skipped -- needs a Hailo-10H device and model.hef")
elif not os.path.exists(WTE):
    print("skipped -- needs workdir/wte.npy from the walkthrough")
else:
    print(f"==> base-scope greedy generation ({HEF})")
    !python "{DIAG_DIR}/generate_base_scope.py" --hef "{HEF}" --wte "{WTE}" --net-scope {NET_SCOPE} --seq {SEQ} --hidden {HIDDEN} --vocab {VOCAB} --n-heads {NHEAD} --n-kv-heads {NKVHEAD} --head-dim {HD} --rope-theta {ROPE_THETA}

In [ ]:
# Prerequisite: the tap-capture reference npz described in the section C
# intro. Flip to True once you have built it on this machine.
RUN_TBT_PROBE = False

if not HAVE_DEVICE:
    print("skipped -- needs a Hailo-10H device and model.hef")
elif not RUN_TBT_PROBE:
    print("skipped -- set RUN_TBT_PROBE = True after building the tap-reference npz")
    print("         (see runtime/diagnostics/README.md)")
else:
    print(f"==> manual prefill + tbt probe ({HEF})")
    !python "{DIAG_DIR}/manual_prefill_tbt_test.py" --hef "{HEF}" --reference "{HF_REFS}" --net-scope {NET_SCOPE} --seq {SEQ} --prefill {PREFILL_SIZE} --hidden {HIDDEN} --vocab {VOCAB} --n-heads {NHEAD} --n-kv-heads {NKVHEAD} --head-dim {HD} --rope-theta {ROPE_THETA}